# Advanced Prompting Patterns: Voting, Branching, and Structured Output

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Implement **Self-Consistency Voting** to filter out one-off reasoning mistakes.
2. Build a **Tree-of-Thought** pipeline that branches, evaluates, and solves.
3. Write a **Structured Output** parser with automatic retry and self-correction.
4. Choose the right pattern based on cost, reliability, and use case.

---

## Why These Patterns Matter

A single Chain-of-Thought pass is fast but fragile. These three patterns each address
a different failure mode:

| Failure Mode | Pattern | Fix |
|---|---|---|
| LLM makes a random arithmetic mistake | Self-Consistency | Run N times, majority vote |
| LLM commits to a bad plan early | Tree-of-Thought | Explore branches before committing |
| LLM returns broken JSON | Structured Output + Retry | Parse, detect error, feed it back |

## Section 1: Setup

In [1]:
import os
import json
import re
from collections import Counter
from openai import AzureOpenAI
from dotenv import load_dotenv

# ── Load credentials from .env ──────────────────────────────────────────────
load_dotenv()

client = AzureOpenAI(
    api_version="2024-12-01-preview",
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
)

MODEL = os.getenv("AZURE_OPENAI_MODEL", "gpt-4.1-mini")

print(f"Model: {MODEL}")
print("Client ready.")

Model: gpt-4.1-mini
Client ready.


---

## Section 2: Self-Consistency Voting

### The Problem with Single-Pass CoT

A single Chain-of-Thought run can produce the wrong answer due to arithmetic slips,
misread details, or unlucky reasoning paths. The error is invisible because the
reasoning *looks* confident.

**Self-Consistency** fixes this: run the same prompt N times with `temperature > 0`,
extract the final verdict from each run, and take the majority vote. When temperature
is above zero, each run takes a slightly different reasoning path. Random errors
average out; correct answers reinforce each other.

### Architecture

```
Question --> [Run 1 (temp=0.7)] --> Answer A
         --> [Run 2 (temp=0.7)] --> Answer A
         --> [Run 3 (temp=0.7)] --> Answer B   --> Majority Vote --> Final: A
         --> [Run 4 (temp=0.7)] --> Answer A
         --> [Run 5 (temp=0.7)] --> Answer A
```

The cost is N x the base cost, but for high-stakes factual answers the accuracy
improvement is worth it.

### Define the Problem

A customer disputes a bill. The model must check arithmetic and render a verdict.
This is a good test case because small rounding or addition errors change the outcome.

In [2]:
billing_question = (
    "You are a billing analyst resolving customer complaints.\n"
    'A client says: "I was charged $275, but my plan is $80/month for 3 months, '
    'and I was supposed to get a $25 activation fee added to the total."\n\n'
    "For each case:\n"
    "1. Identify the amount charged\n"
    "2. Calculate the correct amount\n"
    "3. Compute the difference\n"
    "4. Clearly state if the client was overcharged, undercharged, or correctly charged.\n"
    "Let's think step by step."
)

print(billing_question)

You are a billing analyst resolving customer complaints.
A client says: "I was charged $275, but my plan is $80/month for 3 months, and I was supposed to get a $25 activation fee added to the total."

For each case:
1. Identify the amount charged
2. Calculate the correct amount
3. Compute the difference
4. Clearly state if the client was overcharged, undercharged, or correctly charged.
Let's think step by step.


### Run N Times at Temperature 0.7

In [3]:
NUM_RUNS = 5

billing_responses = []
for i in range(NUM_RUNS):
    answer = client.chat.completions.create(
        model=MODEL,
        temperature=0.7,
        messages=[{"role": "user", "content": billing_question}]
    ).choices[0].message.content
    billing_responses.append(answer)
    print(f"Run {i+1} complete.")

print(f"\nCollected {len(billing_responses)} responses.")

Run 1 complete.


Run 2 complete.


Run 3 complete.


Run 4 complete.


Run 5 complete.

Collected 5 responses.


### Extract Verdicts and Vote

The extraction function scans from the bottom of each response for keywords.
This is intentionally simple -- in production you would use structured output
(covered in Section 4) instead of regex-based extraction.

In [4]:
def extract_verdict(text):
    """Extract billing verdict from model response.
    Scans lines bottom-up for verdict keywords."""
    for line in reversed(text.splitlines()):
        line = line.lower().strip()
        if "correctly" in line:
            return "correctly charged"
        if "overcharged" in line:
            return "overcharged"
        if "undercharged" in line:
            return "undercharged"
    return "unknown"


# Extract verdicts from all runs
billing_votes = [extract_verdict(r) for r in billing_responses]

# Count and determine majority
vote_counts = Counter(billing_votes)
majority = vote_counts.most_common(1)[0][0]

print("Vote Distribution")
print("-" * 40)
for verdict, count in vote_counts.most_common():
    bar = "#" * (count * 6)
    print(f"  {verdict:<20s} {count}/{NUM_RUNS}  {bar}")

print(f"\nMajority vote: {majority}")
print(f"Confidence:    {vote_counts[majority]}/{NUM_RUNS}")

Vote Distribution
----------------------------------------
  overcharged          5/5  ##############################

Majority vote: overcharged
Confidence:    5/5


In [5]:
# Inspect individual reasoning paths
for i, resp in enumerate(billing_responses):
    print(f"\n{'='*60}")
    print(f"Run {i+1} | Verdict: {billing_votes[i]}")
    print(f"{'='*60}")
    print(resp)


Run 1 | Verdict: overcharged
Let's analyze the client's case step by step:

1. **Identify the amount charged:**  
   The client says they were charged **$275**.

2. **Calculate the correct amount:**  
   - The plan is $80/month for 3 months:  
     \( 80 \times 3 = 240 \)  
   - Activation fee is $25 (added to the total):  
     \( 240 + 25 = 265 \)  
   So, the correct amount should be **$265**.

3. **Compute the difference:**  
   Charged amount - Correct amount = \( 275 - 265 = 10 \)

4. **Determine if overcharged, undercharged, or correct:**  
   The client was charged $10 more than the correct amount, so they were **overcharged by $10**.

---

**Summary:**  
- Amount charged: $275  
- Correct amount: $265  
- Difference: $10  
- Conclusion: Client was **overcharged by $10**.

Run 2 | Verdict: overcharged
Let's analyze the client's statement step by step:

### 1. Identify the amount charged
- The client says they were charged **$275**.

### 2. Calculate the correct amount
- Plan c

> **Observation**: The correct total is $80 x 3 + $25 = $265. The client was charged
> $275, so they were overcharged by $10. Self-consistency helps surface the correct
> answer even if one or two runs make arithmetic mistakes.
>
> See the full standalone implementation in `self_consistency.py`.

---

## Section 3: Tree-of-Thought

### Beyond Linear Chains

Standard Chain-of-Thought follows a single linear path. If the first step is wrong,
everything that follows is wasted. Tree-of-Thought (ToT) fixes this by exploring
multiple approaches *before* committing to one.

The pipeline has three stages:

1. **Branch** -- Generate N distinct approaches (high temperature for diversity).
2. **Evaluate** -- Score each branch, prune dead ends (low temperature for consistency).
3. **Solve** -- Fully solve the best branch (low temperature for accuracy).

### Architecture

```
Problem --> [Generate 3 Branches] --> Branch A, Branch B, Branch C
                                           |
                                     [Evaluate & Score]
                                           |
                                     Best Branch --> [Solve Fully] --> Solution
```

Cost is approximately 3x the base cost (one call per stage).

### Define the Problem: Water Jug Puzzle

Classic planning puzzle with multiple valid paths and dead ends -- ideal for ToT.

In [6]:
puzzle = (
    "You have a 4L jug and a 3L jug with no markings. "
    "You need to measure exactly 2L of water. "
    "You can fill, empty, or pour between jugs."
)

print(f"Puzzle: {puzzle}")

Puzzle: You have a 4L jug and a 3L jug with no markings. You need to measure exactly 2L of water. You can fill, empty, or pour between jugs.


### Step 1: Generate Branches (temp=0.7 for diversity)

In [7]:
branch_prompt = f"""Problem: {puzzle}

Generate 3 different possible approaches (branches) to solve this.
For each branch, describe the first 2-3 moves only.
Label them Branch A, Branch B, and Branch C.
Do not solve them fully yet -- just outline the strategy."""

branch_response = client.chat.completions.create(
    model=MODEL,
    temperature=0.7,
    messages=[{"role": "user", "content": branch_prompt}]
)
branches = branch_response.choices[0].message.content

print("STEP 1 -- GENERATE BRANCHES")
print("(exploring multiple reasoning paths)")
print("-" * 50)
print(branches)

STEP 1 -- GENERATE BRANCHES
(exploring multiple reasoning paths)
--------------------------------------------------
Certainly! Here are three different approaches (branches) to start measuring exactly 2L using a 4L and a 3L jug:

---

**Branch A: Fill the 3L jug first**

1. Fill the 3L jug completely (3L).
2. Pour from the 3L jug into the 4L jug until the 4L jug is full or the 3L jug is empty.
3. (Next step would depend on the remaining water in the 3L jug and 4L jug.)

---

**Branch B: Fill the 4L jug first**

1. Fill the 4L jug completely (4L).
2. Pour from the 4L jug into the 3L jug until the 3L jug is full.
3. (Next step would involve the leftover water in the 4L jug.)

---

**Branch C: Start by emptying the 4L jug and using the 3L jug to transfer**

1. Ensure both jugs are empty initially.
2. Fill the 3L jug completely.
3. Pour from the 3L jug into the 4L jug.

---

Each branch starts differently and will explore different ways to isolate exactly 2L. Let me know if you want to con

### Step 2: Evaluate Branches (temp=0 for consistent scoring)

In [8]:
evaluate_prompt = f"""Problem: {puzzle}

Here are 3 proposed approaches:
{branches}

Evaluate each branch:
- Will it lead to a valid solution?
- How many steps will it likely take?
- Are there any dead ends or inefficiencies?

Score each branch 1-10 and clearly state which ones to PRUNE (discard)
and which one is the MOST PROMISING. Explain your reasoning."""

evaluate_response = client.chat.completions.create(
    model=MODEL,
    temperature=0,
    messages=[{"role": "user", "content": evaluate_prompt}]
)
evaluation = evaluate_response.choices[0].message.content

print("STEP 2 -- EVALUATE & PRUNE")
print("(score each branch, discard dead ends)")
print("-" * 50)
print(evaluation)

STEP 2 -- EVALUATE & PRUNE
(score each branch, discard dead ends)
--------------------------------------------------
Let's evaluate each branch carefully based on whether it leads to a valid solution (exactly 2L), how many steps it might take, and any inefficiencies or dead ends.

---

### Branch A: Fill the 3L jug first

**Steps so far:**
1. Fill 3L jug (3L).
2. Pour from 3L jug into 4L jug (4L jug now has 3L, 3L jug is empty).

**Next steps:**
- Fill 3L jug again (3L).
- Pour from 3L jug into 4L jug until 4L jug is full (4L jug currently has 3L, so it can take 1L more).
- After pouring 1L, 3L jug will have 2L left.
- Now, 3L jug has exactly 2L — solution found!

**Evaluation:**
- This is a classic and well-known approach.
- Takes about 4 steps total.
- No dead ends.
- Efficient and straightforward.

**Score:** 9/10

---

### Branch B: Fill the 4L jug first

**Steps so far:**
1. Fill 4L jug (4L).
2. Pour from 4L jug into 3L jug until 3L jug is full (3L jug now has 3L, 4L jug has 1L le

### Step 3: Solve Best Branch (temp=0 for accuracy)

In [9]:
solve_prompt = f"""Problem: {puzzle}

Here are the proposed approaches:
{branches}

Here is the evaluation:
{evaluation}

Now take the MOST PROMISING branch and solve it fully step by step.
Show each move clearly (fill, empty, pour) and the state of both jugs after each move.
Format: Move N: [action] -> 4L jug: XL | 3L jug: YL"""

solve_response = client.chat.completions.create(
    model=MODEL,
    temperature=0,
    messages=[{"role": "user", "content": solve_prompt}]
)
solution = solve_response.choices[0].message.content

print("STEP 3 -- SOLVE BEST BRANCH")
print("(fully solve the most promising path)")
print("-" * 50)
print(solution)

STEP 3 -- SOLVE BEST BRANCH
(fully solve the most promising path)
--------------------------------------------------
Certainly! Let's continue solving from **Branch A** step-by-step, showing each move clearly.

---

### Initial state:
- 4L jug: 0L
- 3L jug: 0L

---

**Move 1:** Fill the 3L jug completely  
-> 4L jug: 0L | 3L jug: 3L

---

**Move 2:** Pour from 3L jug into 4L jug  
(4L jug can hold 4L, currently 0L, so it takes all 3L)  
-> 4L jug: 3L | 3L jug: 0L

---

**Move 3:** Fill the 3L jug completely again  
-> 4L jug: 3L | 3L jug: 3L

---

**Move 4:** Pour from 3L jug into 4L jug until 4L jug is full  
(4L jug currently has 3L, can take 1L more)  
Pour 1L from 3L jug to 4L jug  
-> 4L jug: 4L | 3L jug: 2L

---

**Result:**  
3L jug now has exactly **2L** — the desired amount!

---

### Summary of moves:

| Move | Action                          | 4L jug | 3L jug |
|-------|--------------------------------|--------|--------|
| 1     | Fill 3L jug                    | 0L     | 3L

> **Observation**: ToT mimics deliberate planning by generating alternatives,
> evaluating and pruning bad paths, then committing to and solving the best one.
> Unlike CoT which follows one linear chain, ToT explores the solution space
> before committing.
>
> See the full standalone implementation in `tree_of_thought.py`.

---

## Section 4: Structured Output with Retry

### The JSON Problem

LLMs frequently break JSON format in subtle ways:

- Wrapping output in markdown fences (` ```json ... ``` `)
- Adding explanation text before or after the JSON
- Producing malformed JSON (trailing commas, unquoted keys)

The defense is a two-layer approach:

1. **Parse helper** that handles common edge cases.
2. **Retry loop** that feeds parse errors back to the model and asks it to self-correct.

This is the same observe-correct loop used in ReAct, applied to output formatting.

### Parse Helper

In [10]:
def parse_json_response(text):
    """Parse JSON from model response, handling 3 common edge cases.

    Edge case 1: Model wraps output in ```json ... ```
    Edge case 2: Model adds text before or after the JSON object
    Edge case 3: Model returns malformed JSON

    Returns:
        (parsed_dict, None) on success
        (None, error_string) on failure
    """
    # Edge case 1: strip markdown fences
    if "```" in text:
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]

    # Edge case 2: extract the JSON object from surrounding text
    start = text.find("{")
    end = text.rfind("}") + 1
    if start == -1 or end == 0:
        return None, "No JSON object found in response"

    # Edge case 3: catch malformed JSON
    try:
        return json.loads(text[start:end]), None
    except json.JSONDecodeError as e:
        return None, f"Invalid JSON: {e}"


# Quick test
test_cases = [
    '{"name": "Alice", "age": 30}',                    # clean
    '```json\n{"name": "Bob", "age": 25}\n```',         # markdown fences
    'Here is the result: {"name": "Carol", "age": 28}',  # text before
    'this is not json at all',                            # no JSON
]

for tc in test_cases:
    parsed, error = parse_json_response(tc)
    status = f"OK: {parsed}" if parsed else f"FAIL: {error}"
    print(f"  Input: {tc[:50]:<50s} -> {status}")

  Input: {"name": "Alice", "age": 30}                       -> OK: {'name': 'Alice', 'age': 30}
  Input: ```json
{"name": "Bob", "age": 25}
```             -> OK: {'name': 'Bob', 'age': 25}
  Input: Here is the result: {"name": "Carol", "age": 28}   -> OK: {'name': 'Carol', 'age': 28}
  Input: this is not json at all                            -> FAIL: No JSON object found in response


### Retry Loop with Error Feedback

In [11]:
def prompt_with_retry(messages, max_retries=3):
    """Send a prompt and retry on JSON parse failure.

    On failure, appends the model's broken response and the parse error
    back into the conversation, asking the model to self-correct.

    Returns:
        (raw_text, parsed_dict, None) on success
        (raw_text, None, error_string) after max_retries
    """
    conversation = messages.copy()

    for attempt in range(1, max_retries + 1):
        print(f"  Attempt {attempt}...")

        response = client.chat.completions.create(
            model=MODEL,
            messages=conversation
        )
        raw = response.choices[0].message.content
        parsed, error = parse_json_response(raw)

        # Success
        if parsed:
            return raw, parsed, None

        # Failure -- feed error back and ask model to fix
        print(f"  Failed: {error} -- asking model to fix...")
        conversation.append({"role": "assistant", "content": raw})
        conversation.append({
            "role": "user",
            "content": (
                f"Your response could not be parsed as JSON.\n"
                f"Error: {error}\n"
                f"Your response was: {raw}\n\n"
                f"Return ONLY a valid JSON object. No markdown, no explanation, just raw JSON."
            )
        })

    return raw, None, f"Failed after {max_retries} attempts"


print("prompt_with_retry defined.")

prompt_with_retry defined.


### Task 1: Person Info Extraction

In [12]:
print("=" * 60)
print("TASK 1 -- PERSON INFO EXTRACTION")
print("=" * 60)

raw1, parsed1, error1 = prompt_with_retry([
    {"role": "user", "content": (
        'Extract the information from the text and return ONLY a JSON object '
        'with no extra text.\n'
        'Format: {"name": "...", "age": ..., "city": "..."}\n'
        'Text: John Smith is 28 years old and lives in New York.'
    )}
])

print(f"\nRaw response:\n{raw1}")
if error1:
    print(f"\nFailed to parse: {error1}")
else:
    print(f"\nParsed successfully:")
    print(f"  Name: {parsed1.get('name')}")
    print(f"  Age:  {parsed1.get('age')}")
    print(f"  City: {parsed1.get('city')}")

TASK 1 -- PERSON INFO EXTRACTION
  Attempt 1...



Raw response:
{"name": "John Smith", "age": 28, "city": "New York"}

Parsed successfully:
  Name: John Smith
  Age:  28
  City: New York


### Task 2: Product Review Classification

In [13]:
print("=" * 60)
print("TASK 2 -- PRODUCT REVIEW ANALYSIS")
print("=" * 60)

raw2, parsed2, error2 = prompt_with_retry([
    {"role": "user", "content": (
        'Analyze the following product review and return ONLY a JSON object.\n'
        'Format: {"sentiment": "positive/negative/neutral", "score": 1-10, '
        '"key_issues": [...]}\n'
        'Review: The laptop is fast and looks great but the battery life is '
        'disappointing and it runs hot.'
    )}
])

print(f"\nRaw response:\n{raw2}")
if error2:
    print(f"\nFailed to parse: {error2}")
else:
    print(f"\nParsed successfully:")
    print(f"  Sentiment:  {parsed2.get('sentiment')}")
    print(f"  Score:      {parsed2.get('score')}")
    print(f"  Key issues: {parsed2.get('key_issues')}")

TASK 2 -- PRODUCT REVIEW ANALYSIS
  Attempt 1...



Raw response:
{"sentiment": "neutral", "score": 6, "key_issues": ["battery life", "overheating"]}

Parsed successfully:
  Sentiment:  neutral
  Score:      6
  Key issues: ['battery life', 'overheating']


> **Observation**: The parse helper handles the three most common edge cases.
> The retry loop feeds errors back to the model, leveraging the same
> observe-correct pattern used in ReAct agents.
>
> See the full standalone implementation in `structured_output.py`.

---

## Section 5: When to Use Which

| Pattern | Best For | Cost | Key Mechanism |
|---|---|---|---|
| Self-Consistency | High-stakes factual answers | N x base cost | Temperature diversity + voting |
| Tree-of-Thought | Planning/search problems | 3 x base cost | Branch -> Evaluate -> Solve |
| Structured Output | Any downstream pipeline | 1-3 x base cost | Parse + retry + self-correction |

**Decision guide:**

- Need a single correct number or verdict? Use **Self-Consistency**.
- Problem has multiple strategies and dead ends? Use **Tree-of-Thought**.
- Output feeds into code (API, database, UI)? Use **Structured Output + Retry**.
- These patterns compose: you can run ToT inside a self-consistency loop, or
  require structured JSON output from each ToT branch.

---

## Section 6: Summary and Exercises

### Key Takeaways

| Concept | What It Does | When to Reach for It |
|---|---|---|
| Self-Consistency | Runs N passes, majority vote | Arithmetic, classification, factual Q&A |
| Tree-of-Thought | Branch, evaluate, solve | Puzzles, planning, multi-step search |
| Structured Output + Retry | Parse JSON, feed errors back | Any LLM output consumed by code |
| Temperature tuning | Controls reasoning diversity | Higher for branching, zero for evaluation |

### Exercises

**(a) Self-Consistency at Scale**

Implement self-consistency for a math word problem using 10 runs instead of 5.
Plot the vote distribution. At what N does the majority stabilize?

**(b) ToT for Scheduling**

Design a Tree-of-Thought prompt for a meeting scheduling problem: 4 people,
3 available time slots each, minimize conflicts. Write the branch, evaluate,
and solve prompts.

**(c) Pydantic Validation in the Retry Loop**

Extend `prompt_with_retry` to validate parsed JSON against a Pydantic model.
Feed both JSON parse errors *and* validation errors back to the model.
Test with a schema that requires specific field types and value ranges.

---

**Standalone scripts**: `self_consistency.py`, `tree_of_thought.py`, `structured_output.py`